# SMILE Explainability Analysis: Teacher vs Student Models

**MSc Project: Trustworthy SLMs for Ambient Clinical Scribing**  
**Author: Alireza Rashidi**

This notebook applies the SMILE (Statistical Model-agnostic Interpretability with Local Explanations) method
to compare how different models focus on input words when generating responses.

**Models compared (7 total):**
- Teacher: GPT-4o-mini (via OpenAI API)
- Phi-3.5-mini: Base + Fine-tuned
- Llama-3.2-3B: Base + Fine-tuned
- Llama-3.2-1B: Base + Fine-tuned

**SMILE Algorithm (Aslansefat et al., 2023):**
1. Generate perturbed versions of the input by randomly dropping words
2. Query each model with original + perturbed inputs
3. Measure semantic distance (WMD) between original and perturbed outputs
4. Fit weighted linear regression: perturbation mask → output similarity
5. Regression coefficients = word importance scores

**References:**
- Aslansefat et al. (2023) IEEE Software - SMILE
- Kusner et al. (2015) ICML - Word Mover's Distance
- Ribeiro et al. (2016) KDD - LIME

## 1. Setup & Dependencies

In [ ]:
# Install dependencies (run once)
# !pip install openai sentence-transformers gensim pot scikit-learn matplotlib numpy
# Note: For local models you also need: transformers, torch, unsloth

In [ ]:
import os
import numpy as np
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib import transforms
from sklearn.linear_model import LinearRegression
from gensim.models import KeyedVectors

# Windows compatibility for local models
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"

print("Imports complete.")

## 2. Configuration

Set your paths and API keys here.

In [ ]:
# ============================================================
# CONFIGURE THESE PATHS
# ============================================================

# OpenAI API key for GPT-4o-mini teacher
os.environ["OPENAI_API_KEY"] = "your-api-key-here"  # Or set in environment

# Word2Vec model for WMD computation
# Download from: https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM/edit
WORD2VEC_PATH = "D:/models/GoogleNews-vectors-negative300.bin"

# Local model paths (hf_merged directories - READ ONLY)
MODEL_CONFIGS = {
    "Phi-3.5 FT": {
        "path": "./checkpoints/phi35_clinical_scribe/hf_merged",
        "type": "local",
    },
    "Phi-3.5 Base": {
        "path": "unsloth/Phi-3.5-mini-instruct",
        "type": "local",
    },
    "Llama-3B FT": {
        "path": "./checkpoints/llama32_3b_clinical_scribe/hf_merged",
        "type": "local",
    },
    "Llama-3B Base": {
        "path": "unsloth/Llama-3.2-3B-Instruct",
        "type": "local",
    },
    "Llama-1B FT": {
        "path": "./checkpoints/llama32_1b_clinical_scribe/hf_merged",
        "type": "local",
    },
    "Llama-1B Base": {
        "path": "unsloth/Llama-3.2-1B-Instruct",
        "type": "local",
    },
    "GPT-4o-mini (Teacher)": {
        "path": "gpt-4o-mini",
        "type": "openai",
    },
}

# Test sentences for SMILE analysis
TEST_PROMPTS = [
    "What is the meaning of life?",
    "What are the common symptoms of diabetes?",
    "How should hypertension be managed in elderly patients?",
]

# SMILE parameters
NUM_PERTURBATIONS = 64
SEED = 1024
KERNEL_WIDTH = 0.25
MAX_TOKENS = 100

print(f"Configured {len(MODEL_CONFIGS)} models and {len(TEST_PROMPTS)} test prompts.")

## 3. Model Query Functions

Unified interface for querying both OpenAI API (teacher) and local HuggingFace models (students).

In [ ]:
# ============================================================
# OpenAI API query (for GPT-4o-mini teacher)
# ============================================================

def query_openai(text, model="gpt-4o-mini", max_tokens=100):
    """Query OpenAI chat completion API."""
    from openai import OpenAI
    client = OpenAI()
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": text}],
            max_tokens=max_tokens,
            temperature=0,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {e}"


# ============================================================
# Local model query (for fine-tuned and base SLMs)
# ============================================================

# Cache to avoid reloading models multiple times
_model_cache = {}

def load_local_model(model_path, max_seq_length=4096):
    """Load a local HuggingFace model. Cached after first load."""
    if model_path in _model_cache:
        return _model_cache[model_path]
    
    print(f"  Loading model: {model_path}...")
    
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch
    
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    model.eval()
    
    _model_cache[model_path] = (model, tokenizer)
    print(f"  Loaded: {model_path}")
    return model, tokenizer


def query_local(text, model_path, max_tokens=100):
    """Query a local HuggingFace model."""
    import torch
    
    model, tokenizer = load_local_model(model_path)
    
    # Build chat-format input
    messages = [{"role": "user", "content": text}]
    
    if hasattr(tokenizer, 'apply_chat_template'):
        input_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        input_text = text
    
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.1,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    generated_tokens = outputs[0][input_len:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    return response


# ============================================================
# Unified query function
# ============================================================

def query_model(text, model_name, model_config):
    """Query any model using the appropriate backend."""
    if model_config["type"] == "openai":
        return query_openai(text, model=model_config["path"], max_tokens=MAX_TOKENS)
    else:
        return query_local(text, model_config["path"], max_tokens=MAX_TOKENS)


print("Query functions defined.")

## 4. SMILE Core Algorithm

The SMILE method (Aslansefat et al., 2023):
1. **Perturb** the input by randomly including/excluding words
2. **Query** the model with each perturbation
3. **Measure** semantic distance between original and perturbed outputs (WMD)
4. **Regress** perturbation masks → output similarity with distance-based kernel weights
5. **Coefficients** = per-word importance scores

In [ ]:
# Load Word2Vec model for WMD computation
print("Loading Word2Vec model (this takes ~1 minute)...")
w2v_model = KeyedVectors.load_word2vec_format(WORD2VEC_PATH, binary=True)
print(f"Word2Vec loaded: {len(w2v_model)} words")

In [ ]:
def generate_perturbations(words, num_perturb=64, seed=1024):
    """
    Generate random binary perturbation masks for the input words.
    Each mask is a tuple of 0/1 indicating whether each word is included.
    
    Returns:
        perturbations: list of tuples (binary masks)
        perturbed_texts: list of strings (perturbed prompts)
    """
    rng = np.random.default_rng(seed)
    num_words = len(words)
    
    unique_perturbations = set()
    perturbations = []
    perturbed_texts = []
    
    attempts = 0
    max_attempts = num_perturb * 10
    
    while len(unique_perturbations) < num_perturb and attempts < max_attempts:
        p = tuple(rng.binomial(1, 0.5, size=num_words))
        if p not in unique_perturbations and sum(p) > 0:
            unique_perturbations.add(p)
            # Select words where mask=1
            selected = [w for w, flag in zip(words, p) if flag == 1]
            if len(selected) < 2:
                available = [w for w in words if w.strip()]
                extra = rng.choice(available, 2, replace=False)
                selected = list(set(selected + list(extra)))
            
            perturbations.append(p)
            perturbed_texts.append(' '.join(selected))
        attempts += 1
    
    # Fill remaining with repeats if needed
    while len(perturbations) < num_perturb:
        p = rng.choice(list(unique_perturbations))
        selected = [w for w, flag in zip(words, p) if flag == 1]
        perturbations.append(tuple(p))
        perturbed_texts.append(' '.join(selected))
    
    return perturbations, perturbed_texts


def compute_smile_scores(prompt, model_name, model_config, w2v_model,
                         num_perturb=64, seed=1024, kernel_width=0.25):
    """
    Run the full SMILE pipeline for a single model and prompt.
    
    Returns:
        words: list of input words
        coefficients: numpy array of per-word importance scores
        metadata: dict with intermediate results
    """
    words = prompt.split()
    
    # Step 1: Get original output
    print(f"    Querying original prompt...")
    original_output = query_model(prompt, model_name, model_config)
    print(f"    Original output: {original_output[:100]}...")
    
    # Step 2: Generate perturbations
    perturbations, perturbed_texts = generate_perturbations(words, num_perturb, seed)
    
    # Step 3: Query model with each perturbation
    print(f"    Querying {len(perturbed_texts)} perturbations...")
    perturbed_outputs = []
    for i, pt in enumerate(perturbed_texts):
        output = query_model(pt, model_name, model_config)
        perturbed_outputs.append(output)
        if (i + 1) % 16 == 0:
            print(f"      {i+1}/{len(perturbed_texts)} done")
    
    # Step 4: Compute WMD between original output and each perturbed output
    epsilon = 1e-8
    wmd_output_scores = []
    for po in perturbed_outputs:
        try:
            dist = w2v_model.wmdistance(original_output.split(), po.split())
            if np.isinf(dist):
                dist = 10.0  # Cap infinite distances
        except Exception:
            dist = 10.0
        wmd_output_scores.append(dist)
    
    # Convert WMD to similarity (inverse, scaled to [0,1])
    inverse_wmd = [1.0 / (d + epsilon) for d in wmd_output_scores]
    min_inv = min(inverse_wmd)
    max_inv = max(inverse_wmd)
    range_inv = max_inv - min_inv if max_inv > min_inv else 1.0
    similarities = [(v - min_inv) / range_inv for v in inverse_wmd]
    
    # Step 5: Compute WMD between original prompt and each perturbed prompt
    wmd_prompt_scores = []
    for pt in perturbed_texts:
        try:
            dist = w2v_model.wmdistance(prompt.split(), pt.split())
            if np.isinf(dist):
                dist = 10.0
        except Exception:
            dist = 10.0
        wmd_prompt_scores.append(dist)
    
    # Step 6: Compute kernel weights (locality)
    distance_values = np.array(wmd_prompt_scores)
    weights = np.sqrt(np.exp(-(distance_values ** 2) / (kernel_width ** 2)))
    
    # Step 7: Fit weighted linear regression
    X = np.vstack(perturbations)
    y = np.array(similarities)
    
    linear_model = LinearRegression()
    linear_model.fit(X=X, y=y, sample_weight=weights)
    coefficients = linear_model.coef_
    
    print(f"    Coefficients: {dict(zip(words, coefficients))}")
    
    metadata = {
        "original_output": original_output,
        "num_perturbations": len(perturbations),
        "r_squared": linear_model.score(X, y, sample_weight=weights),
    }
    
    return words, coefficients, metadata


print("SMILE functions defined.")

## 5. Heatmap Visualisation

In [ ]:
def plot_text_heatmap(words, scores, title="", width=12, height=0.6,
                      max_word_per_line=15, word_spacing=20,
                      score_fontsize=10, save_path=None):
    """
    Plot a heatmap over words based on SMILE importance scores.
    Blue = high positive importance, Red = high negative importance, White = neutral.
    """
    fig = plt.figure(figsize=(width, height))
    ax = plt.gca()
    ax.set_title(title, loc='left', fontsize=12, fontweight='bold')
    cmap = plt.cm.ScalarMappable(cmap=plt.cm.bwr)
    cmap.set_clim(0, 1)
    canvas = ax.figure.canvas
    t = ax.transData
    
    max_abs = np.max(np.abs(scores))
    if max_abs == 0:
        max_abs = 1.0
    normalized_scores = 0.5 * scores / max_abs + 0.5
    
    loc_y = -0.2
    for i, (token, score) in enumerate(zip(words, scores)):
        *rgb, _ = cmap.to_rgba(normalized_scores[i], bytes=True)
        color = '#%02x%02x%02x' % tuple(rgb)
        text = ax.text(0.0, loc_y, token,
                       bbox={'facecolor': color, 'pad': 5.0, 'linewidth': 1,
                             'boxstyle': 'round,pad=0.5'},
                       transform=t, fontsize=14)
        text.draw(canvas.get_renderer())
        ex = text.get_window_extent()
        score_text = ax.text(0.01, loc_y - 1, f"{score:.3f}",
                             transform=t, fontsize=score_fontsize, ha='center')
        score_text.draw(canvas.get_renderer())
        if (i + 1) % max_word_per_line == 0:
            loc_y -= 2.5
            t = ax.transData
        else:
            t = transforms.offset_copy(text._transform, x=ex.width + word_spacing, units='dots')
    
    ax.axis('off')
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
    plt.show()


def plot_comparison_bar(all_results, prompt, save_path=None):
    """
    Side-by-side bar chart comparing word importance across all models.
    """
    words = prompt.split()
    model_names = list(all_results.keys())
    n_models = len(model_names)
    n_words = len(words)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(n_words)
    w = 0.8 / n_models
    colors = plt.cm.Set2(np.linspace(0, 1, max(n_models, 3)))
    
    for i, model_name in enumerate(model_names):
        coeffs = all_results[model_name]["coefficients"]
        ax.bar(x + i * w, coeffs, w, label=model_name, color=colors[i],
               edgecolor='black', linewidth=0.3)
    
    ax.set_xlabel('Input Word', fontsize=12)
    ax.set_ylabel('SMILE Importance Score', fontsize=12)
    ax.set_title(f'SMILE Word Importance Comparison\nPrompt: "{prompt}"',
                 fontsize=13, fontweight='bold')
    ax.set_xticks(x + w * (n_models - 1) / 2)
    ax.set_xticklabels(words, fontsize=11)
    ax.legend(fontsize=8, ncol=2, loc='upper left')
    ax.grid(axis='y', alpha=0.3)
    ax.axhline(y=0, color='black', linewidth=0.5)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


print("Visualisation functions defined.")

## 6. Run SMILE Analysis

Run one model at a time to manage GPU memory. After running a local model, you can free its memory before loading the next.

In [ ]:
import gc
import torch

def free_gpu_memory():
    """Free GPU memory between model runs."""
    global _model_cache
    _model_cache.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("  GPU memory freed.")

In [ ]:
# ============================================================
# SELECT WHICH MODELS AND PROMPTS TO RUN
# ============================================================

# Run all models on the first prompt by default.
# Change these to control what runs:
MODELS_TO_RUN = list(MODEL_CONFIGS.keys())  # All 7 models
PROMPTS_TO_RUN = TEST_PROMPTS[:1]  # Start with first prompt; expand later

print(f"Will run {len(MODELS_TO_RUN)} models on {len(PROMPTS_TO_RUN)} prompts")
print(f"Total API/inference calls: ~{len(MODELS_TO_RUN) * len(PROMPTS_TO_RUN) * (NUM_PERTURBATIONS + 1)}")

In [ ]:
# ============================================================
# MAIN ANALYSIS LOOP
# ============================================================

all_results = {}  # {prompt: {model_name: {words, coefficients, metadata}}}

for prompt in PROMPTS_TO_RUN:
    print(f"\n{'='*70}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*70}")
    
    prompt_results = {}
    
    for model_name in MODELS_TO_RUN:
        config = MODEL_CONFIGS[model_name]
        print(f"\n  --- {model_name} ({config['type']}) ---")
        
        try:
            start = time.time()
            words, coefficients, metadata = compute_smile_scores(
                prompt, model_name, config, w2v_model,
                num_perturb=NUM_PERTURBATIONS, seed=SEED, kernel_width=KERNEL_WIDTH,
            )
            elapsed = time.time() - start
            
            prompt_results[model_name] = {
                "words": words,
                "coefficients": coefficients,
                "metadata": metadata,
                "time_seconds": elapsed,
            }
            print(f"    Completed in {elapsed:.1f}s (R²={metadata['r_squared']:.3f})")
            
        except Exception as e:
            print(f"    ERROR: {e}")
            prompt_results[model_name] = {"error": str(e)}
        
        # Free GPU memory between local models
        if config['type'] == 'local':
            free_gpu_memory()
    
    all_results[prompt] = prompt_results

print("\n\nAll analyses complete!")

## 7. Visualise Results

In [ ]:
# ============================================================
# HEATMAPS: One per model per prompt
# ============================================================

output_dir = Path("./smile_results")
output_dir.mkdir(exist_ok=True)

for prompt, prompt_results in all_results.items():
    print(f"\nPrompt: {prompt}")
    print("=" * 60)
    
    for model_name, result in prompt_results.items():
        if "error" in result:
            print(f"  {model_name}: SKIPPED (error)")
            continue
        
        words = result["words"]
        coeffs = result["coefficients"]
        
        safe_name = model_name.replace(' ', '_').replace('(', '').replace(')', '').replace('-', '')
        safe_prompt = prompt[:30].replace(' ', '_').replace('?', '')
        save_path = output_dir / f"heatmap_{safe_name}_{safe_prompt}.png"
        
        plot_text_heatmap(
            words, coeffs,
            title=f"{model_name}",
            save_path=str(save_path),
        )
        
        print(f"  {model_name}: {dict(zip(words, np.round(coeffs, 3)))}")

In [ ]:
# ============================================================
# COMPARISON BAR CHARTS: All models on same plot
# ============================================================

for prompt, prompt_results in all_results.items():
    # Filter out errors
    valid_results = {k: v for k, v in prompt_results.items() if "error" not in v}
    
    if len(valid_results) >= 2:
        safe_prompt = prompt[:30].replace(' ', '_').replace('?', '')
        save_path = output_dir / f"comparison_{safe_prompt}.png"
        plot_comparison_bar(valid_results, prompt, save_path=str(save_path))

## 8. Quantitative Comparison

Compute correlation between models' word importance rankings to see if they focus on the same words.

In [ ]:
from scipy.stats import spearmanr, pearsonr

for prompt, prompt_results in all_results.items():
    print(f"\nPrompt: \"{prompt}\"")
    print("=" * 70)
    
    valid = {k: v for k, v in prompt_results.items() if "error" not in v}
    model_names = list(valid.keys())
    
    if len(model_names) < 2:
        print("  Not enough models for comparison.")
        continue
    
    # Build coefficient matrix
    words = valid[model_names[0]]["words"]
    
    # Print summary table
    print(f"\n  {'Word':<15}", end="")
    for mn in model_names:
        short = mn[:12]
        print(f"{short:>13}", end="")
    print()
    print("  " + "-" * (15 + 13 * len(model_names)))
    
    for wi, word in enumerate(words):
        print(f"  {word:<15}", end="")
        for mn in model_names:
            c = valid[mn]["coefficients"][wi]
            print(f"{c:>13.4f}", end="")
        print()
    
    # Pairwise correlations
    print(f"\n  Pairwise Spearman Correlations:")
    print(f"  {'':20}", end="")
    for mn in model_names:
        print(f"{mn[:12]:>13}", end="")
    print()
    
    for i, mn_i in enumerate(model_names):
        print(f"  {mn_i[:20]:<20}", end="")
        for j, mn_j in enumerate(model_names):
            if i == j:
                print(f"{'1.000':>13}", end="")
            else:
                c_i = valid[mn_i]["coefficients"]
                c_j = valid[mn_j]["coefficients"]
                rho, _ = spearmanr(c_i, c_j)
                print(f"{rho:>13.3f}", end="")
        print()
    
    # Highlight: Teacher vs Fine-tuned comparison
    teacher_key = [k for k in model_names if "Teacher" in k or "GPT" in k]
    ft_keys = [k for k in model_names if "FT" in k]
    
    if teacher_key and ft_keys:
        print(f"\n  Teacher vs Fine-tuned Student Correlations:")
        t_coeffs = valid[teacher_key[0]]["coefficients"]
        for fk in ft_keys:
            s_coeffs = valid[fk]["coefficients"]
            rho, p = spearmanr(t_coeffs, s_coeffs)
            print(f"    {teacher_key[0]} vs {fk}: rho={rho:.3f} (p={p:.4f})")
    
    # Highlight: Base vs Fine-tuned comparison
    base_keys = [k for k in model_names if "Base" in k]
    if base_keys and ft_keys:
        print(f"\n  Base vs Fine-tuned (same architecture) Correlations:")
        for bk in base_keys:
            arch = bk.split(" Base")[0]
            matching_ft = [fk for fk in ft_keys if arch in fk]
            if matching_ft:
                b_coeffs = valid[bk]["coefficients"]
                f_coeffs = valid[matching_ft[0]]["coefficients"]
                rho, p = spearmanr(b_coeffs, f_coeffs)
                print(f"    {bk} vs {matching_ft[0]}: rho={rho:.3f} (p={p:.4f})")

## 9. Save Results

In [ ]:
# Save all results to JSON for later analysis
save_data = {}
for prompt, prompt_results in all_results.items():
    save_data[prompt] = {}
    for model_name, result in prompt_results.items():
        if "error" in result:
            save_data[prompt][model_name] = {"error": result["error"]}
        else:
            save_data[prompt][model_name] = {
                "words": result["words"],
                "coefficients": result["coefficients"].tolist(),
                "r_squared": result["metadata"]["r_squared"],
                "original_output": result["metadata"]["original_output"][:500],
                "time_seconds": result.get("time_seconds", 0),
            }

with open(output_dir / "smile_results.json", "w") as f:
    json.dump(save_data, f, indent=2)

print(f"Results saved to {output_dir / 'smile_results.json'}")

## 10. Discussion

### Key Questions to Investigate:

1. **Do the teacher and fine-tuned students focus on the same words?**
   - High Spearman correlation between Teacher and FT models → knowledge distillation preserved attention patterns
   - Low correlation → students learned different strategies

2. **Does fine-tuning change word importance?**
   - Compare Base vs FT for same architecture
   - If clinical prompts show different focus after FT → fine-tuning shifted model attention toward clinically relevant terms

3. **Do smaller models focus differently?**
   - Compare 1B vs 3B vs 3.8B → does model size affect which words matter most?

4. **General vs Clinical prompts:**
   - The general prompt ("meaning of life") tests baseline language understanding
   - Clinical prompts test domain-specific attention patterns
   - FT models should show more clinical focus on medical prompts

### References

1. Aslansefat, K. et al. (2023). *Explaining Black Boxes with a SMILE*. IEEE Software.
2. Kusner, M. et al. (2015). *From Word Embeddings to Document Distances*. ICML.
3. Ribeiro, M. T. et al. (2016). *"Why Should I Trust You?": Explaining the Predictions of Any Classifier*. KDD.